In [1]:
import h5py
import numpy as np
## to see the structure of the hdf5 file and what keys are available
path = "./dataset/ICs_sampling_0.hdf5"
with h5py.File(path, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP] {name}")
    f.visititems(show)

[DATASET] APs shape=(40000, 2) dtype=float64
[DATASET] LANs shape=(40000, 2) dtype=float64
[DATASET] aratios shape=(40000, 1) dtype=float64
[DATASET] e_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] eccs shape=(40000, 2) dtype=float64
[DATASET] h_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] incs shape=(40000, 2) dtype=float64
[DATASET] masses shape=(40000, 3) dtype=float64
[DATASET] mratios shape=(40000, 2) dtype=float64
[DATASET] periods shape=(40000, 2) dtype=float64
[DATASET] positions shape=(40000, 3, 3) dtype=float64
[DATASET] q_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] rel_positions shape=(40000, 2, 3) dtype=float64
[DATASET] rel_velocities shape=(40000, 2, 3) dtype=float64
[DATASET] smaxes shape=(40000, 2) dtype=float64
[DATASET] tAnos shape=(40000, 2) dtype=float64
[DATASET] velocities shape=(40000, 3, 3) dtype=float64


In [2]:
# To regroup the data into tokens of size 8, we can reshape the arrays accordingly. 
input_path = "./dataset/ICs_sampling_0.hdf5"
output_path = "./dataset/ICs_sampling_0_groupof8.hdf5"

token_size = 8
tokenized = {}


with h5py.File(input_path, "r") as fin:
    for name, obj in fin.items():
        if not isinstance(obj, h5py.Dataset):
            continue
        x = obj[:]
        n = (x.shape[0] // token_size) * token_size
        x_cut = x[:n]
        x_tok = x_cut.reshape(-1, token_size, *x.shape[1:])
        tokenized[name] = x_tok
        print(f"{name}: {x.shape} -> {x_tok.shape}")
        
with h5py.File(output_path, "w") as fout:
    for name, arr in tokenized.items():
        fout.create_dataset(name, data=arr)

print("Token regroup done")
# print(tokenized["APs"])

APs: (40000, 2) -> (5000, 8, 2)
LANs: (40000, 2) -> (5000, 8, 2)
aratios: (40000, 1) -> (5000, 8, 1)
e_uvecs: (40000, 2, 3) -> (5000, 8, 2, 3)
eccs: (40000, 2) -> (5000, 8, 2)
h_uvecs: (40000, 2, 3) -> (5000, 8, 2, 3)
incs: (40000, 2) -> (5000, 8, 2)
masses: (40000, 3) -> (5000, 8, 3)
mratios: (40000, 2) -> (5000, 8, 2)
periods: (40000, 2) -> (5000, 8, 2)
positions: (40000, 3, 3) -> (5000, 8, 3, 3)
q_uvecs: (40000, 2, 3) -> (5000, 8, 2, 3)
rel_positions: (40000, 2, 3) -> (5000, 8, 2, 3)
rel_velocities: (40000, 2, 3) -> (5000, 8, 2, 3)
smaxes: (40000, 2) -> (5000, 8, 2)
tAnos: (40000, 2) -> (5000, 8, 2)
velocities: (40000, 3, 3) -> (5000, 8, 3, 3)
Token regroup done


In [3]:
import pandas as pd
import numpy as np
import torch

df = pd.read_csv("dataset/completed_systems_sampling_0.txt", sep=r"\s+", header=None,
                 names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"])

# datatype
df = df.astype({"index":"int64","num_batch":"int64","time_unbound":"float32","ignore_other_than-1":"float32","dignosis":"int64"})
df = df.sort_values("index")  # sort by index column
df["label"] = np.where(df["time_unbound"] >= 5000, "unbound", "bound")  # create binary label based on time_unbound
df["group_8"] = np.arange(len(df)) // 8
prob_each_group = (
    df.groupby("group_8")["label"]
      .apply(lambda x: (x == "unbound").mean())   
      .reset_index(name="prob_unbound")
)
print(df.columns)
print(df.head(10))
print(prob_each_group)
#tensor = torch.from_numpy(arr)  
#print(tensor.shape, tensor.dtype)


Index(['index', 'num_batch', 'time_unbound', 'ignore_other_than-1', 'dignosis',
       'label', 'group_8'],
      dtype='str')
      index  num_batch  time_unbound  ignore_other_than-1  dignosis    label  \
2394      0          0        5000.0            65.464226         0  unbound   
2309      1          0        5000.0            63.461426         0  unbound   
2320      2          0        5000.0            63.699547         0  unbound   
2316      3          0        5000.0            63.610538         0  unbound   
2339      4          0        5000.0            64.137978         0  unbound   
2322      5          0        5000.0            63.498493         0  unbound   
2361      6          0        5000.0            64.632912         0  unbound   
2304      7          0        5000.0            63.364311         0  unbound   
166       8          0          55.0             7.035079         0    bound   
338       9          0          45.0            11.731630         0    bo

In [7]:
import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1) Y: prob_unbound per group
# -----------------------------
def build_prob_unbound_labels(
    labels_path="dataset/completed_systems_sampling_0.txt",
    token_size=8,
    threshold_time=5000.0,
):
    df = pd.read_csv(
        labels_path, sep=r"\s+", header=None,
        names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"]
    )
    df = df.astype({
        "index":"int64","num_batch":"int64","time_unbound":"float32",
        "ignore_other_than-1":"float32","dignosis":"int64"
    })
    df = df.sort_values("index").reset_index(drop=True)

    n = (len(df) // token_size) * token_size
    df = df.iloc[:n].copy()

    df["label"] = (df["time_unbound"] >= threshold_time).astype(np.float32)
    df["group_8"] = np.arange(len(df)) // token_size

    y_prob = df.groupby("group_8")["label"].mean().to_numpy(dtype=np.float32)  # (N_groups,)
    return y_prob


# -----------------------------
# 2) X: take only 1 IC per group
#    from (N, 8, ...) -> (N, ...)
# -----------------------------
def load_tokenized_X_take_one(
    h5_path="dataset/ICs_sampling_0_groupof8.hdf5",
    keys=None,                 # None = all datasets
    take_token_idx=0,          # take the 0-th IC in each group
    dtype=np.float32,
):
    with h5py.File(h5_path, "r") as f:
        all_keys = [k for k in f.keys() if isinstance(f[k], h5py.Dataset)]
        if keys is None:
            keys = all_keys
        else:
            missing = [k for k in keys if k not in f]
            if missing:
                raise KeyError(f"Missing keys in hdf5: {missing}. Available: {all_keys}")

        X_parts = []
        N = None
        for k in keys:
            arr = f[k][:]
            # expect (N, 8, ...)
            if arr.ndim < 2:
                raise ValueError(f"{k} has shape {arr.shape}, expected at least 2D (N,8,...)")
            if N is None:
                N = arr.shape[0]
            elif arr.shape[0] != N:
                raise ValueError(f"{k} has N={arr.shape[0]} but expected N={N}")
            if arr.shape[1] <= take_token_idx:
                raise ValueError(f"{k} second dim is {arr.shape[1]}, cannot take idx={take_token_idx}")

            one = arr[:, take_token_idx, ...].astype(dtype, copy=False)   # (N, ...)
            one_flat = one.reshape(N, -1)                                 # (N, Dk)
            X_parts.append(one_flat)

        X = np.concatenate(X_parts, axis=1)  # (N, D_total)
    return X, keys


def train_test_split_groups(X, y_prob, test_ratio=0.2, seed=42, stratify=True):
    assert X.shape[0] == y_prob.shape[0]
    N = X.shape[0]
    rng = np.random.default_rng(seed)
    idx = np.arange(N)

    if not stratify:
        rng.shuffle(idx)
        n_test = int(round(N * test_ratio))
        return idx[n_test:], idx[:n_test]

    # approximate stratify for continuous y_prob: {0, (0,1), 1}
    bins = np.where(y_prob == 0.0, 0, np.where(y_prob == 1.0, 2, 1))
    train_idx_list, test_idx_list = [], []
    for b in np.unique(bins):
        b_idx = idx[bins == b]
        rng.shuffle(b_idx)
        n_test_b = int(round(len(b_idx) * test_ratio))
        test_idx_list.append(b_idx[:n_test_b])
        train_idx_list.append(b_idx[n_test_b:])
    train_idx = np.concatenate(train_idx_list)
    test_idx = np.concatenate(test_idx_list)
    rng.shuffle(train_idx); rng.shuffle(test_idx)
    return train_idx, test_idx


def standardize_train_test(X_train, X_test, eps=1e-6):
    mu = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.maximum(std, eps)
    return (X_train - mu) / std, (X_test - mu) / std, mu, std


def make_loaders(X_train, y_train, X_test, y_test, batch_size=128):
    X_train_t = torch.from_numpy(X_train).float()
    y_train_t = torch.from_numpy(y_train).float().unsqueeze(1)  # (N,1)
    X_test_t  = torch.from_numpy(X_test).float()
    y_test_t  = torch.from_numpy(y_test).float().unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


# =============================
# MAIN
# =============================
y_prob = build_prob_unbound_labels("dataset/completed_systems_sampling_0.txt", token_size=8, threshold_time=5000.0)
X, used_keys = load_tokenized_X_take_one("dataset/ICs_sampling_0_groupof8.hdf5", keys=None, take_token_idx=0)

# align
N = min(X.shape[0], y_prob.shape[0])
X = X[:N]
y_prob = y_prob[:N]

print("X:", X.shape, "y_prob:", y_prob.shape, "keys_used:", len(used_keys))

train_idx, test_idx = train_test_split_groups(X, y_prob, test_ratio=0.2, seed=42, stratify=True)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_prob[train_idx], y_prob[test_idx]

X_train, X_test, mu, std = standardize_train_test(X_train, X_test)

train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test, batch_size=128)

print("Train:", X_train.shape, y_train.shape, " Test:", X_test.shape, y_test.shape)
print("y_train prob stats:", float(y_train.min()), float(y_train.mean()), float(y_train.max()))
print("y_test  prob stats:", float(y_test.min()), float(y_test.mean()), float(y_test.max()))


X: (5000, 68) y_prob: (5000,) keys_used: 17
Train: (4000, 68) (4000,)  Test: (1000, 68) (1000,)
y_train prob stats: 0.0 0.35603123903274536 1.0
y_test  prob stats: 0.0 0.3636249899864197 1.0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLPProb(nn.Module):
    """
    Predict probability p(unbound) in [0,1] from flattened initial condition features.
    """
    def __init__(self, in_dim: int, hidden=(512, 256, 128), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers += [nn.Linear(prev, 1)]  # logits
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        logits = self.net(x)                 # (B,1)
        prob = torch.sigmoid(logits)         # (B,1) in [0,1]
        return prob


def train_mlp_prob(
    train_loader,
    test_loader,
    in_dim: int,
    device=None,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=30,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MLPProb(in_dim=in_dim).to(device)

    # Since y is a probability in [0,1], MSE is a clean baseline.
    # You can also try: nn.BCELoss() (on prob) or BCEWithLogitsLoss (on logits).
    loss_fn = nn.MSELoss()

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for ep in range(1, epochs + 1):
        # ---- train ----
        model.train()
        train_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)                 # (B,1) prob
            loss = loss_fn(pred, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            train_loss += loss.item() * bs
            n_train += bs

        train_loss /= max(n_train, 1)

        # ---- eval ----
        model.eval()
        test_loss = 0.0
        mae = 0.0
        n_test = 0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                bs = xb.size(0)
                test_loss += loss.item() * bs
                mae += torch.abs(pred - yb).sum().item()
                n_test += bs

        test_loss /= max(n_test, 1)
        mae /= max(n_test, 1)

        print(f"epoch {ep:03d} | train_mse {train_loss:.6f} | test_mse {test_loss:.6f} | test_mae {mae:.6f}")

    return model

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = X_train.shape[1]
model = train_mlp_prob(train_loader, test_loader, in_dim=in_dim, device=device, epochs=200)

epoch 001 | train_mse 0.140649 | test_mse 0.132086 | test_mae 0.279217
epoch 002 | train_mse 0.118421 | test_mse 0.123794 | test_mae 0.274409
epoch 003 | train_mse 0.106888 | test_mse 0.119143 | test_mae 0.266264
epoch 004 | train_mse 0.098726 | test_mse 0.116441 | test_mae 0.257889
epoch 005 | train_mse 0.089110 | test_mse 0.110034 | test_mae 0.247761
epoch 006 | train_mse 0.084687 | test_mse 0.109652 | test_mae 0.245526
epoch 007 | train_mse 0.079031 | test_mse 0.110053 | test_mae 0.241301
epoch 008 | train_mse 0.077468 | test_mse 0.110159 | test_mae 0.235266
epoch 009 | train_mse 0.073009 | test_mse 0.103912 | test_mae 0.229001
epoch 010 | train_mse 0.067762 | test_mse 0.107419 | test_mae 0.235209
epoch 011 | train_mse 0.068786 | test_mse 0.102302 | test_mae 0.233242
epoch 012 | train_mse 0.064792 | test_mse 0.105980 | test_mae 0.233501
epoch 013 | train_mse 0.059117 | test_mse 0.106173 | test_mae 0.225649
epoch 014 | train_mse 0.059271 | test_mse 0.110658 | test_mae 0.232291
epoch 

In [15]:
import h5py
import numpy as np

def load_natural_tokens_from_raw_h5(
    h5_path="dataset/ICs_sampling_0.hdf5",
    token_size=8,
    take_within_group=0,   # 取每组第几个（0~7都一样的话随便）
    dtype=np.float32,
):
    """
    Return:
      X_tokens: (N, T, 3) padded to 3 dims
      token_names: list[str] length T
    """
    with h5py.File(h5_path, "r") as f:
        # N_systems = 40000/8 = 5000
        n_rows = next(iter(f.values())).shape[0]
        n_groups = n_rows // token_size
        idx = np.arange(n_groups) * token_size + take_within_group  # e.g. 0,8,16,...

        tokens = []
        names = []

        def add_token(arr_1d, name):
            """arr_1d shape (N, d<=3) -> pad to (N,3)"""
            arr_1d = arr_1d.astype(dtype, copy=False)
            if arr_1d.ndim != 2:
                raise ValueError(f"{name} expected 2D (N,d), got {arr_1d.shape}")
            N, d = arr_1d.shape
            if d > 3:
                raise ValueError(f"{name} has d={d} > 3; handle separately")
            pad = np.zeros((N, 3), dtype=dtype)
            pad[:, :d] = arr_1d
            tokens.append(pad)
            names.append(name)

        def add_tokens_from_matrix(arr_2d, base_name):
            """
            arr_2d shape (N, k, 3) -> k tokens each (N,3)
            """
            arr_2d = arr_2d.astype(dtype, copy=False)
            if arr_2d.ndim != 3 or arr_2d.shape[-1] != 3:
                raise ValueError(f"{base_name} expected (N,k,3), got {arr_2d.shape}")
            N, k, _ = arr_2d.shape
            for i in range(k):
                tokens.append(arr_2d[:, i, :])
                names.append(f"{base_name}[{i}]")

        # ---- scalar-ish / small vectors ----
        add_token(f["APs"][idx, :], "APs")               # (N,2)
        add_token(f["LANs"][idx, :], "LANs")             # (N,2)
        add_token(f["aratios"][idx, :], "aratios")       # (N,1)
        add_token(f["eccs"][idx, :], "eccs")             # (N,2)
        add_token(f["incs"][idx, :], "incs")             # (N,2)
        add_token(f["masses"][idx, :], "masses")         # (N,3)
        add_token(f["mratios"][idx, :], "mratios")       # (N,2)
        add_token(f["periods"][idx, :], "periods")       # (N,2)
        add_token(f["smaxes"][idx, :], "smaxes")         # (N,2)
        add_token(f["tAnos"][idx, :], "tAnos")           # (N,2)

        # ---- (N,2,3) -> 2 tokens ----
        add_tokens_from_matrix(f["e_uvecs"][idx, :, :], "e_uvecs")
        add_tokens_from_matrix(f["h_uvecs"][idx, :, :], "h_uvecs")
        add_tokens_from_matrix(f["q_uvecs"][idx, :, :], "q_uvecs")
        add_tokens_from_matrix(f["rel_positions"][idx, :, :], "rel_positions")
        add_tokens_from_matrix(f["rel_velocities"][idx, :, :], "rel_velocities")

        # ---- (N,3,3) -> 3 tokens ----
        add_tokens_from_matrix(f["positions"][idx, :, :], "positions")
        add_tokens_from_matrix(f["velocities"][idx, :, :], "velocities")

        # Stack: list of (N,3) -> (N,T,3)
        X_tokens = np.stack(tokens, axis=1)  # (N, T, 3)
        return X_tokens, names

In [20]:
import copy
import torch
import torch.nn as nn

class NaturalTokenTransformer(nn.Module):
    def __init__(self, n_tokens: int, d_in: int = 3, d_model: int = 128,
                 nhead: int = 8, num_layers: int = 4, dim_ff: int = 256,
                 dropout: float = 0.15):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, 1 + n_tokens, d_model))
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)

        # 把每个 token (3维) 投影到 d_model
        self.in_proj = nn.Linear(d_in, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)  # logits

    def forward(self, x_tok):
        # x_tok: (B, T, 3)
        z = self.in_proj(x_tok)  # (B,T,d_model)
        B, T, _ = z.shape
        cls = self.cls.expand(B, -1, -1)   # (B,1,d_model)
        z = torch.cat([cls, z], dim=1)     # (B,1+T,d_model)
        z = z + self.pos[:, : (1 + T), :]
        z = self.encoder(z)
        z = self.norm(z)
        pooled = z[:, 0, :]                # CLS
        logits = self.head(pooled)         # (B,1)
        return logits


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    loss_fn = nn.BCEWithLogitsLoss()
    total_loss, total_mae, n = 0.0, 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        prob = torch.sigmoid(logits)
        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_mae += torch.abs(prob - yb).sum().item()
        n += bs
    return total_loss / n, total_mae / n


def train_transformer_natural_tokens(train_loader, test_loader, n_tokens,
                                     device=None, epochs=200, lr=3e-4,
                                     weight_decay=1e-2, patience=50):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = NaturalTokenTransformer(n_tokens=n_tokens).to(device)

    loss_fn = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_loss = float("inf")
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, n = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = xb.size(0)
            total_loss += loss.item() * bs
            n += bs

        train_loss = total_loss / max(n, 1)
        test_loss, test_mae = eval_epoch(model, test_loader, device)

        if test_loss < best_loss - 1e-5:
            best_loss = test_loss
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1

        print(f"epoch {ep:03d} | train_loss {train_loss:.6f} | test_loss {test_loss:.6f} | test_mae {test_mae:.6f} | bad {bad}/{patience}")

        if bad >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# X_tokens: (N, T, 3)
X_tokens, token_names = load_natural_tokens_from_raw_h5("dataset/ICs_sampling_0.hdf5", token_size=8)
N, T, Din = X_tokens.shape
print("X_tokens:", X_tokens.shape, "T:", T)
print("First few tokens:", token_names[:5], "...")

# y_prob: (N,) 你用你之前 build_prob_unbound_labels 那段即可
# y_prob = ...

# 对齐
N2 = min(len(y_prob), N)
X_tokens = X_tokens[:N2]
y_prob = y_prob[:N2]

# split（你原来的 train_idx/test_idx 逻辑继续用）
# train_idx, test_idx = ...

X_train = X_tokens[train_idx]
X_test  = X_tokens[test_idx]
y_train = y_prob[train_idx]
y_test  = y_prob[test_idx]

# 转 tensor
X_train_t = torch.from_numpy(X_train).float()
X_test_t  = torch.from_numpy(X_test).float()
y_train_t = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_t  = torch.from_numpy(y_test).float().unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=128, shuffle=False)

# 训练
model = train_transformer_natural_tokens(train_loader, test_loader, n_tokens=T)

X_tokens: (5000, 26, 3) T: 26
First few tokens: ['APs', 'LANs', 'aratios', 'eccs', 'incs'] ...


/tmp/ipykernel_13887/2166891216.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 001 | train_loss 0.649829 | test_loss 0.646748 | test_mae 0.377849 | bad 0/50
epoch 002 | train_loss 0.633879 | test_loss 0.621874 | test_mae 0.351956 | bad 0/50
epoch 003 | train_loss 0.613350 | test_loss 0.602981 | test_mae 0.342684 | bad 0/50
epoch 004 | train_loss 0.598812 | test_loss 0.665000 | test_mae 0.302497 | bad 1/50
epoch 005 | train_loss 0.571074 | test_loss 0.559397 | test_mae 0.302840 | bad 0/50
epoch 006 | train_loss 0.552267 | test_loss 0.550754 | test_mae 0.285285 | bad 0/50
epoch 007 | train_loss 0.541390 | test_loss 0.541078 | test_mae 0.281520 | bad 0/50
epoch 008 | train_loss 0.530494 | test_loss 0.545414 | test_mae 0.289567 | bad 1/50
epoch 009 | train_loss 0.527754 | test_loss 0.546351 | test_mae 0.274960 | bad 2/50
epoch 010 | train_loss 0.536259 | test_loss 0.553646 | test_mae 0.275079 | bad 3/50
epoch 011 | train_loss 0.521023 | test_loss 0.536138 | test_mae 0.276388 | bad 0/50
epoch 012 | train_loss 0.525722 | test_loss 0.570090 | test_mae 0.273357 | b